# Notebook 6. Where the coupling settles (Fig. 6)

Reproduces Fig. 6. Reads no data and writes the panels of Fig. 6 to the folder `figs`. Run time about 18 minutes. `QUICK = True` in the first code cell shortens the runs for a test.

In [1]:
QUICK = False
T_RUN = 3000      # the run read by Fig. 4 and the copying analysis of Fig. S2
T_POP = 5000      # the four conditions, the hypothesis tests, the sweeps and Fig. 5; the same seeds,
                  # so the first 3,000 steps of these runs are the runs of Fig. 4

import numpy as np
import matplotlib.pyplot as plt
import networkx as nx
from math import log2

plt.rcParams.update({'figure.dpi': 110, 'font.size': 8, 'axes.spines.top': False,
                     'axes.spines.right': False, 'axes.linewidth': 0.7,
                     'xtick.major.width': 0.7, 'ytick.major.width': 0.7,
                     'legend.frameon': False, 'font.family': 'DejaVu Sans'})

CEIL = log2(100)          # 6.6439 bits, the even-coupling ceiling at N = 100
BA_KAPPA = 10.10          # matched scale-free ensemble, control-entropy paper Table 1
BA_DH_DEG = 0.357         # its degree-weighted compression
BA_RDEG = -0.17           # its degree assortativity
N = 100
COL = {'baseline': '#7A7A7A', 'explore': '#2E7D4E', 'imitate': '#2E5090', 'both': '#A4161A'}
CONDS = {'baseline': dict(pr=0.0, pe=0.0), 'explore': dict(pr=0.0, pe=0.05),
         'imitate': dict(pr=0.3, pe=0.0), 'both': dict(pr=0.3, pe=0.05)}

def seeds(n):
    return list(range(n if not QUICK else max(2, n // 5)))

print('ceiling log2 N =', round(CEIL, 4), 'bits')

ceiling log2 N = 6.6439 bits


## Step 0. The model

In [2]:
import numpy as np
import networkx as nx

LOG2 = np.log(2.0)


# ----------------------------------------------------------------------
# measures (definitions of the second study)
# ----------------------------------------------------------------------
def control_entropy(v):
    """Control entropy in bits of a non-negative vector of shares (Eq. 11)."""
    v = np.asarray(v, float)
    tot = v.sum()
    if tot <= 0:
        return 0.0
    p = v / tot
    p = p[p > 0]
    return float(-(p * np.log2(p)).sum())


def eigen_shares(W):
    """Shares from the leading eigenvector of a symmetric coupling matrix."""
    vals, vecs = np.linalg.eigh(W)
    v = np.abs(vecs[:, -1])
    return v / v.sum()


def adjacency_measures(A, W):
    """Measures read from the unweighted structure, computed as in Table 1 of the
    control-entropy paper: eigenvector weighting on the largest connected component,
    degree assortativity on the whole graph."""
    G = nx.from_numpy_array(A.astype(int))
    deg = A.sum(axis=1)
    s = W.sum(axis=1)
    iu, ju = np.nonzero(np.triu(A, 1))

    def edge_corr(v):
        x, y = np.concatenate([v[iu], v[ju]]), np.concatenate([v[ju], v[iu]])
        return float(np.corrcoef(x, y)[0, 1]) if x.std() > 1e-12 else float('nan')

    r_deg, r_str = edge_corr(deg.astype(float)), edge_corr(s)
    if not nx.is_connected(G):
        G = G.subgraph(max(nx.connected_components(G), key=len)).copy()
    c = nx.eigenvector_centrality_numpy(G)
    v = np.abs(np.array([c[n] for n in G.nodes()]))
    return dict(H_adj=control_entropy(v), r_deg=float(r_deg), r_str=r_str, gc=G.number_of_nodes())


# ----------------------------------------------------------------------
# the five rules, one function each
# ----------------------------------------------------------------------
def ring_lattice(N, k):
    """Regular ring lattice: every element coupled to its k nearest neighbours."""
    return nx.to_numpy_array(nx.watts_strogatz_graph(N, k, 0.0), dtype=bool)


def softmax_allocation(A, beta, shat):
    """Eq 1. Element i splits one unit of coupling over its partners j with weights
    exp(beta_i * shat_j). beta_i = 0 gives even coupling."""
    L = np.where(A, beta[:, None] * shat[None, :], -np.inf)
    L = L - L.max(axis=1, keepdims=True)
    E = np.where(A, np.exp(L), 0.0)
    return E / E.sum(axis=1, keepdims=True)


def commitment(phi0, phi1, C, fm):
    """Eq 2. Commitment of an element, in the linear form of the published model,
    f = phi0 + phi1 * C, truncated to [0, 1 - fm]."""
    return np.clip(phi0 + phi1 * C, 0.0, 1.0 - fm)


def protection(f, c, pmax, chalf):
    """Eq 8. Saturating protection bought by committed capability f * c."""
    x = f * c
    return np.where(x > 0, pmax / (1.0 + chalf / np.maximum(x, 1e-12)), 0.0)


def fermi(dc, s):
    """Eq 10. Probability of copying a partner whose capability is higher by dc."""
    return 1.0 / (1.0 + np.exp(-s * dc))


def logistic(z, g, theta):
    """Response function of an element, used inside the spread (Eq 5)."""
    return 1.0 / (1.0 + np.exp(-g * (z - theta)))


class ElementModel:
    """Network-agent model of element-level exploration and imitation.

    Each element holds
      beta_i        direction of coupling (0 = even coupling, > 0 = toward stronger partners)
      phi0_i, phi1_i commitment strategy, giving f_i = phi0_i + phi1_i * C_i
      c_i           capability, the accumulated stimulation it compares with partners
    Each coupling (i, j) holds a weight w_ij.
    """

    def __init__(self, N=100, k=6, seed=0,
                 pr=0.3, pe=0.05, sigma=0.1, sigma_f=0.01, s=2.0,   # strategy dynamics
                 M=10, rho=0.5, nE=5, switch=None, psi=0.3,        # feedback, two channels
                 gamma=1.0, pn=0.02, pl=0.3, lam=1.0, g=4.0, theta=1.0,   # seeding and spread
                 fm=0.1, pmax=1.0, chalf=0.1, f0=0.1,              # capability and protection
                 eta=0.05, delta=0.01, w0=1.0,                     # consolidation
                 pw=2.0, kmin=2, loss=1.0,                         # pruning and its cost
                 spread=True, two_part=True):
        self.p = dict(N=N, k=k, pr=pr, pe=pe, sigma=sigma, sigma_f=sigma_f, s=s, M=M, rho=rho,
                      nE=nE, switch=switch, psi=psi, gamma=gamma, pn=pn, pl=pl, lam=lam, g=g,
                      theta=theta, fm=fm, pmax=pmax, chalf=chalf, f0=f0, eta=eta, delta=delta,
                      w0=w0, pw=pw, kmin=kmin, loss=loss, spread=spread, two_part=two_part)
        self.rng = np.random.default_rng(seed)
        self.N, self.k = N, k
        self.A = ring_lattice(N, k)
        self.W = self.A.astype(float) * w0
        self.beta = np.zeros(N)
        self.phi0 = np.full(N, f0)
        self.phi1 = np.zeros(N)
        self.c = np.ones(N)
        self.x = np.zeros(N)          # activation of the previous step
        self.v = np.ones(N) / np.sqrt(N)   # leading eigenvector, kept warm
        self.t = 0
        self.n_act = 0.0
        self.n_rounds = 0.0
        self.set_essential(self.rng.choice(N, nE, replace=False))

    # ---------- state helpers ----------
    def set_essential(self, E):
        self.E = np.asarray(E)
        self.isE = np.zeros(self.N, bool)
        self.isE[self.E] = True

    def strengths(self):
        return self.W.sum(axis=1)

    def shat(self):
        s = self.strengths()
        return s / s.max()

    def centrality(self, iters=8):
        """Leading eigenvector of W by power iteration, warm started, scaled to a
        maximum of one. This is the C of the commitment rule (Eq 2)."""
        v = self.v
        for _ in range(iters):
            v = self.W @ v
            n = np.linalg.norm(v)
            if n < 1e-12:
                v = np.ones(self.N) / np.sqrt(self.N)
                break
            v = v / n
        self.v = np.abs(v)
        return self.v / max(self.v.max(), 1e-12)

    def allocation(self):
        return softmax_allocation(self.A, self.beta, self.shat())

    # ---------- one step ----------
    def feedback(self):
        """Eq 3. Two channels of information.
        Process feedback is intrinsic: every element receives afferent input in
        proportion to its own activation in the previous step.
        Result feedback is augmented: M packets, each landing on a task-essential
        element with probability rho and on any element otherwise."""
        p, rng = self.p, self.rng
        u = p['psi'] * self.x.copy()
        if p['M'] > 0:
            to_E = rng.random(p['M']) < p['rho']
            tgt = np.where(to_E, rng.choice(self.E, p['M']), rng.integers(0, self.N, p['M']))
            u = u + np.bincount(tgt, minlength=self.N).astype(float)
        return u

    def spread(self, u, a):
        """Eq 4 and Eq 5. Stimulus is seeded where information arrives and then
        travels along the couplings.
        Seeding: element i is stimulated with probability 1 - (1 - p_n) exp(-gamma u_i),
        so an element fires spontaneously at rate p_n and more often when information
        reaches it.
        Spread: an element already stimulated passes activation to partner j with
        the coupling it allocates, and j responds with probability
        pl * logistic(g (d_j - theta)). Each element activates at most once per step.
        The cascade is run to completion inside the step."""
        p, rng, N = self.p, self.rng, self.N
        act = rng.random(N) < (1.0 - (1.0 - p['pn']) * np.exp(-p['gamma'] * u))   # Eq 4
        if not p['spread']:
            self.n_act, self.n_rounds = float(act.sum()), 0.0
            return act
        frontier = act.copy()
        rounds = 0
        while frontier.any() and rounds < 20:
            # drive reaching each candidate through the allocation of active elements
            d = p['lam'] * self.k * (frontier @ a)
            prob = p['pl'] * logistic(d, p['g'], p['theta'])       # Eq 5
            new = (rng.random(N) < prob) & ~act
            act |= new
            frontier = new
            rounds += 1
        self.n_act, self.n_rounds = float(act.sum()), float(rounds)
        return act

    def step(self):
        p, rng, N = self.p, self.rng, self.N
        if p['switch'] and self.t > 0 and self.t % p['switch'] == 0:
            self.set_essential(rng.choice(N, p['nE'], replace=False))

        C = self.centrality()
        a = self.allocation()                                     # Eq 1
        f = commitment(self.phi0, self.phi1, C, p['fm']) if p['two_part'] else np.zeros(N)  # Eq 2
        u = self.feedback()                                       # Eq 3
        act = self.spread(u, a)                                   # Eq 4, Eq 5
        x = act.astype(float)
        self.x = x

        # Eq 6. Payoff is the stimulation the element gathers. It reaches partners
        # through its own allocation and it integrates in proportion to the coupling
        # it holds, so both the direction and the amount of coupling enter.
        s = self.strengths()
        pay = (s / s.mean()) * (a @ x)
        self.c = pay + np.clip(1.0 - p['fm'] - f, 0.0, None) * self.c   # Eq 7

        # Eq 8 and Eq 9. Commitment protects a coupling against decay.
        P = protection(f, self.c, p['pmax'], p['chalf'])
        Pij = 0.5 * (P[:, None] + P[None, :])
        abar = 0.5 * (a + a.T)
        self.W = np.where(self.A,
                          p['w0'] + (1 - p['delta'] * (1 - Pij)) * (self.W - p['w0'])
                          + p['eta'] * self.k * abar * np.outer(x, x), 0.0)

        # Eq S1. Long-term change: a coupling falls out of use, and unless the
        # capability committed to it holds it, it is lost and replaced elsewhere.
        n_rw = int(p['pw']) + (1 if rng.random() < p['pw'] - int(p['pw']) else 0)
        for _ in range(n_rw):
            self._rewire(P)

        # Eq 10. Imitation along a coupling, synchronous, two independent passes.
        b_old, f0_old, f1_old, c_old = self.beta.copy(), self.phi0.copy(), self.phi1.copy(), self.c.copy()
        for i in np.flatnonzero(rng.random(N) < p['pr']):
            nb = np.flatnonzero(self.A[i])
            j = nb[rng.integers(nb.size)]
            if rng.random() < fermi(c_old[j] - c_old[i], p['s']):
                self.beta[i] = b_old[j]
        if p['two_part']:
            for i in np.flatnonzero(rng.random(N) < p['pr']):
                nb = np.flatnonzero(self.A[i])
                j = nb[rng.integers(nb.size)]
                if rng.random() < fermi(c_old[j] - c_old[i], p['s']):
                    self.phi0[i], self.phi1[i] = f0_old[j], f1_old[j]

        # Eq S2. Exploration of both parts of the strategy.
        ex = rng.random(N) < p['pe']
        self.beta[ex] += rng.normal(0.0, p['sigma'], ex.sum())
        if p['two_part']:
            ex0 = rng.random(N) < p['pe']
            self.phi0[ex0] += rng.normal(0.0, p['sigma_f'], ex0.sum())
            ex1 = rng.random(N) < p['pe']
            self.phi1[ex1] += rng.normal(0.0, p['sigma_f'], ex1.sum())
        self.t += 1
        return x

    def _rewire(self, P):
        """One element drops its weakest coupling, unless protection holds it, and
        attaches to a new partner chosen with exp(beta_i * shat_j)."""
        p, rng = self.p, self.rng
        i = rng.integers(self.N)
        deg = self.A.sum(axis=1)
        nb = np.flatnonzero(self.A[i])
        nb = nb[deg[nb] > p['kmin']]
        if deg[i] <= p['kmin'] or nb.size == 0:
            return
        j = nb[np.argmin(self.W[i, nb])]
        if rng.random() < 0.5 * (P[i] + P[j]):     # the coupling is held
            return
        # the route is lost, and with it the capability the two elements had built
        # through it. With loss = 1 this is the capability loss of the published model.
        self.c[i] *= (1.0 - p['loss'])
        self.c[j] *= (1.0 - p['loss'])
        cand = np.flatnonzero(~self.A[i])
        cand = cand[cand != i]
        if cand.size == 0:
            return
        z = self.beta[i] * self.shat()[cand]
        w = np.exp(z - z.max())
        new = rng.choice(cand, p=w / w.sum())
        self.A[i, j] = self.A[j, i] = False
        self.W[i, j] = self.W[j, i] = 0.0
        self.A[i, new] = self.A[new, i] = True
        self.W[i, new] = self.W[new, i] = p['w0']

    # ---------- measures ----------
    def measures(self, adj=False):
        s = self.strengths()
        deg = self.A.sum(axis=1)
        top = np.argsort(s)[-len(self.E):]
        nbE = self.A[self.E].any(axis=0) & ~self.isE
        tot = s.sum()
        C = self.centrality()
        f = commitment(self.phi0, self.phi1, C, self.p['fm'])
        return dict(H_str=control_entropy(s), H_eig=control_entropy(eigen_shares(self.W)),
                    H_deg=control_entropy(deg), kappa=(deg ** 2).mean() / deg.mean(),
                    beta_mean=self.beta.mean(), beta_var=self.beta.var(),
                    f_mean=float(f.mean()), phi0_mean=float(self.phi0.mean()),
                    phi1_mean=float(self.phi1.mean()), c_mean=float(self.c.mean()),
                    act=self.n_act / self.N, rounds=self.n_rounds,
                    hub_align=self.isE[top].mean(),
                    share_E=s[self.isE].sum() / tot, share_nbE=s[nbE].sum() / tot,
                    **(adjacency_measures(self.A, self.W) if adj else {}))

    def run(self, T, every=25, adj=False):
        rec = {}
        for step in range(T):
            self.step()
            if step % every == 0 or step == T - 1:
                m = self.measures(adj=adj)
                m['t'] = self.t
                for key, v in m.items():
                    rec.setdefault(key, []).append(v)
        return {key: np.array(v) for key, v in rec.items()}


def replicate(T, seeds, adj=True, **kw):
    """Run the model for several seeds and return the final measures."""
    out = []
    for sd in seeds:
        m = ElementModel(seed=sd, **kw)
        for _ in range(T):
            m.step()
        out.append(m.measures(adj=adj))
    return {key: np.array([o[key] for o in out]) for key in out[0]}

In [3]:
from scipy.stats import mannwhitneyu

def cliffs_delta(a, b):
    a, b = np.asarray(a), np.asarray(b)
    gt = sum((x > y) for x in a for y in b)
    lt = sum((x < y) for x in a for y in b)
    return (gt - lt) / (len(a) * len(b))

def boot_median(a, reps=5000, seed=0):
    rng2 = np.random.default_rng(seed)
    a = np.asarray(a)
    d = [np.median(rng2.choice(a, len(a))) for _ in range(reps)]
    return np.percentile(d, [2.5, 97.5])

def boot_diff(a, b, reps=5000, seed=0):
    rng2 = np.random.default_rng(seed)
    a, b = np.asarray(a), np.asarray(b)
    d = [np.median(rng2.choice(a, len(a))) - np.median(rng2.choice(b, len(b))) for _ in range(reps)]
    return np.percentile(d, [2.5, 97.5])

def summary(tag, v, fmt='.3f'):
    v = np.asarray(v, float); lo, hi = boot_median(v)
    print(f'{tag}: median {np.median(v):{fmt}} [{lo:{fmt}}, {hi:{fmt}}], '
          f'mean {v.mean():{fmt}}, sd {v.std(ddof=1):{fmt}}, range {v.min():{fmt}} to {v.max():{fmt}}')

def compare(tag, a, b, unit=''):
    u, p = mannwhitneyu(a, b, alternative='two-sided')
    d = cliffs_delta(a, b)
    lo, hi = boot_diff(a, b)
    diff = np.median(a) - np.median(b)
    print(f'{tag}: U = {u:.1f}, p = {p:.4f}, Cliff delta = {d:+.2f}, '
          f'median difference = {diff:+.3f}{unit} [{lo:+.3f}, {hi:+.3f}]')
    return dict(U=u, p=p, d=d, diff=diff, lo=lo, hi=hi)

def star(p):
    return '***' if p < 0.001 else '**' if p < 0.01 else '*' if p < 0.05 else 'n.s.'

## Step 1. The runs, read inside every step

In [4]:
EVERY = 25
T_HALF = T_POP // 2
BINS = np.arange(-8.0, 8.01, 0.5); MID = (BINS[:-1] + BINS[1:]) / 2


def ring_distance(N_, E):
    i = np.arange(N_)[:, None]; d = np.abs(i - np.asarray(E)[None]); d = np.minimum(d, N_ - d)
    return d.min(1)


class Read(ElementModel):
    '''The model with counters. The step below is the step of the model, line for line.'''

    def start_reading(self):
        N_ = self.N
        self.order = np.lexsort((np.arange(N_), ring_distance(N_, self.E)))
        self.r_act = np.zeros(N_); self.r_c = np.zeros(N_); self.r_n = 0
        self.r_draw = np.zeros(len(BINS) - 1); self.r_copy = np.zeros(len(BINS) - 1)
        self.h_toE = np.zeros(len(BINS) - 1); self.k_toE = np.zeros(len(BINS) - 1)
        self.n_other = np.zeros(2)
        self.reading = False

    def budget(self):
        s = self.W.sum(1); tot = s.sum()
        nb = self.A[self.E].any(axis=0) & ~self.isE
        return s[self.isE].sum() / tot, s[nb].sum() / tot, (~self.isE & ~nb)[self.order]

    def step(self):
        p, rng, N_ = self.p, self.rng, self.N
        if p['switch'] and self.t > 0 and self.t % p['switch'] == 0:
            self.set_essential(rng.choice(N_, p['nE'], replace=False))
        C = self.centrality()
        a = self.allocation()
        f = commitment(self.phi0, self.phi1, C, p['fm']) if p['two_part'] else np.zeros(N_)
        u = self.feedback()
        act = self.spread(u, a)
        x = act.astype(float)
        self.x = x
        s = self.strengths()
        pay = (s / s.mean()) * (a @ x)
        self.c = pay + np.clip(1.0 - p['fm'] - f, 0.0, None) * self.c
        P = protection(f, self.c, p['pmax'], p['chalf'])
        Pij = 0.5 * (P[:, None] + P[None, :])
        abar = 0.5 * (a + a.T)
        self.W = np.where(self.A,
                          p['w0'] + (1 - p['delta'] * (1 - Pij)) * (self.W - p['w0'])
                          + p['eta'] * self.k * abar * np.outer(x, x), 0.0)
        n_rw = int(p['pw']) + (1 if rng.random() < p['pw'] - int(p['pw']) else 0)
        for _ in range(n_rw):
            self._rewire(P)
        b_old, f0_old, f1_old, c_old = self.beta.copy(), self.phi0.copy(), self.phi1.copy(), self.c.copy()
        if self.reading:
            self.r_act += x; self.r_c += c_old; self.r_n += 1
        for i in np.flatnonzero(rng.random(N_) < p['pr']):
            nb = np.flatnonzero(self.A[i])
            j = nb[rng.integers(nb.size)]
            dc = c_old[j] - c_old[i]
            copied = rng.random() < fermi(dc, p['s'])
            if copied:
                self.beta[i] = b_old[j]
            if self.reading:
                k = np.searchsorted(BINS, np.clip(dc, BINS[0], BINS[-1] - 1e-9), side='right') - 1
                self.r_draw[k] += 1; self.r_copy[k] += copied
                if not self.isE[i]:
                    if self.isE[j]:
                        self.h_toE[k] += 1; self.k_toE[k] += copied
                    else:
                        self.n_other += (1, copied)
        if p['two_part']:
            for i in np.flatnonzero(rng.random(N_) < p['pr']):
                nb = np.flatnonzero(self.A[i])
                j = nb[rng.integers(nb.size)]
                if rng.random() < fermi(c_old[j] - c_old[i], p['s']):
                    self.phi0[i], self.phi1[i] = f0_old[j], f1_old[j]
        ex = rng.random(N_) < p['pe']
        self.beta[ex] += rng.normal(0.0, p['sigma'], ex.sum())
        if p['two_part']:
            ex0 = rng.random(N_) < p['pe']
            self.phi0[ex0] += rng.normal(0.0, p['sigma_f'], ex0.sum())
            ex1 = rng.random(N_) < p['pe']
            self.phi1[ex1] += rng.normal(0.0, p['sigma_f'], ex1.sum())
        self.t += 1
        return x


def follow(name, seed, keep_model=False):
    m = Read(seed=seed, **RUN_CONDS[name])
    m.start_reading()
    E, P, REST = [], [], []
    def keep():
        e, p_, r = m.budget(); E.append(e); P.append(p_); REST.append(r)
    keep()
    for step in range(1, T_POP + 1):
        m.reading = step > T_HALF
        m.step()
        if step % EVERY == 0:
            keep()
    mm = m.measures(adj=False)
    isE, A, W = m.isE, m.A, m.W
    touchE = A & (isE[:, None] | isE[None, :])
    E, P = np.array(E), np.array(P)
    out = dict(seed=seed, E=E, P=P, R=1.0 - E - P, REST=np.array(REST, dtype=np.uint8), W=W.copy(), isE=isE.copy(),
               same=bool(abs(mm['share_E'] - E[-1]) < 1e-12 and abs(mm['share_nbE'] - P[-1]) < 1e-12),
               act_E=m.r_act[isE].mean() / m.r_n, act_O=m.r_act[~isE].mean() / m.r_n,
               c_E=m.r_c[isE].mean() / m.r_n, c_O=m.r_c[~isE].mean() / m.r_n,
               w_E=W[touchE].mean(), w_O=W[A & ~touchE].mean(), sd_beta=m.beta.std(),
               draw=m.r_draw, copy=m.r_copy, h_toE=m.h_toE, k_toE=m.k_toE, n_other=m.n_other)
    if keep_model:
        out['model'] = m
    return out


RHOS = (0.0, 0.1, 0.25, 0.5, 1.0)
RUN_CONDS = {**CONDS, **{f'rho{r:g}': dict(pr=0.3, pe=0.05, rho=r) for r in RHOS if r != 0.5}}
SERIES = {r: ('both' if r == 0.5 else f'rho{r:g}') for r in RHOS}
NS = seeds(30)
NSRC = min(6, len(NS))
RUNS = {name: [follow(name, sd, keep_model=(name in ('rho0', 'rho1') and sd < NSRC)) for sd in NS]
        for name in RUN_CONDS}
T = np.arange(0, T_POP + 1, EVERY); TMAX = T_POP
print(f'{len(NS)} seeds in each of {len(RUN_CONDS)} conditions, {T_POP} steps, budget read every {EVERY} steps, '
      f'the game read over steps {T_HALF + 1} to {T_POP}')

plain = ElementModel(seed=0, **CONDS['both'])
for _ in range(T_POP):
    plain.step()
CHECK_PLAIN = bool(np.array_equal(plain.W, RUNS['both'][0]['W']))
CHECK_SAME = all(r['same'] for n in RUNS for r in RUNS[n])
print('check, a read run equals the same seed run straight through:', CHECK_PLAIN)
print('check, the budget read from the couplings equals the measures of the model at the last step:', CHECK_SAME)

30 seeds in each of 8 conditions, 5000 steps, budget read every 25 steps, the game read over steps 2501 to 5000
check, a read run equals the same seed run straight through: True
check, the budget read from the couplings equals the measures of the model at the last step: True


## Step 2. The budget at the start and at the end, and where it settles as precision rises

In [5]:
def stack(name, key):
    return np.array([r[key] for r in RUNS[name]])

r0 = RUNS['both'][0]
START = (r0['E'][0], r0['P'][0], r0['R'][0])                  # seed 0, the open circle of panel C
S0 = {n: np.array([[r['E'][0], r['P'][0]] for r in RUNS[n]]) for n in RUNS}
CHECK_START = all(np.array_equal(S0[n], S0['both']) for n in S0)
P0 = S0['both'][:, 1]
START_MED = (float(np.median(S0['both'][:, 0])), float(np.median(P0)), float(np.median(1.0 - S0['both'][:, 0] - P0)))
print(f'start over the {len(NS)} essential sets, medians: task-essential {START_MED[0]:.3f}, recruited '
      f'{START_MED[1]:.3f} (from {P0.min():.3f} to {P0.max():.3f}), unrecruited {START_MED[2]:.3f}')
print(f'start of seed 0, drawn as the open circle of panel C: task-essential {START[0]:.3f}, recruited '
      f'{START[1]:.3f}, unrecruited {START[2]:.3f}')
print('check, every condition starts from the same budget seed by seed:', CHECK_START)
END = {}
print('condition     task-essential           recruited                unrecruited              unrecruited below 0.05')
for name in CONDS:
    e, p, r = stack(name, 'E')[:, -1], stack(name, 'P')[:, -1], stack(name, 'R')[:, -1]
    END[name] = dict(E=e, P=p, R=r)
    cells = []
    for v in (e, p, r):
        lo, hi = boot_median(v)
        cells.append(f'{np.median(v):.3f} [{lo:.3f}, {hi:.3f}]')
    print(f'{name:<12}  ' + '   '.join(f'{c:<22}' for c in cells) + f'   {(r < 0.05).mean():.2f}')
rest3 = lambda key: np.concatenate([END[n][key] for n in ('baseline', 'explore', 'imitate')])
print(f'the other three pooled: task-essential {np.median(rest3("E")):.3f}, recruited {np.median(rest3("P")):.3f}, '
      f'unrecruited {np.median(rest3("R")):.3f}')
T_REST_EXP = compare('unrecruited share, both vs exploration only', END['both']['R'], END['explore']['R'])
T_REST_BASE = compare('unrecruited share, both vs baseline', END['both']['R'], END['baseline']['R'])
T_PART_EXP = compare('recruited share, both vs exploration only', END['both']['P'], END['explore']['P'])

med_R = np.median(stack('both', 'R'), 0)
below = np.nonzero(med_R < 0.05)[0]
T_R05 = int(T[below[0]]) if len(below) else None
med_E = np.median(stack('both', 'E'), 0)
cross = np.nonzero(med_E > 0.2)[0]
T_E02 = int(T[cross[0]]) if len(cross) else None
print(f'combined condition: median unrecruited share below 0.05 from step {T_R05}, '
      f'median essential share above 0.2 from step {T_E02}')

SETTLE = {}
print('precision   settling point at the last step, means: task-essential, recruited, unrecruited')
for rho in RHOS:
    e, p = stack(SERIES[rho], 'E')[:, -1], stack(SERIES[rho], 'P')[:, -1]
    SETTLE[rho] = (e.mean(), p.mean(), 1 - e.mean() - p.mean())
    print(f'rho = {rho:<5}  {SETTLE[rho][0]:.3f}  {SETTLE[rho][1]:.3f}  {SETTLE[rho][2]:.3f}')

import copy as _copy
T_RE = 3000 if not QUICK else 300
RESTART = {}
for target, source in ((0.0, 'rho1'), (1.0, 'rho0')):
    ends = []
    for r in RUNS[source][:NSRC]:
        m = _copy.deepcopy(r['model'])
        m.p['rho'] = target
        m.reading = False
        m.rng = np.random.default_rng(10_000 + r['seed'])
        for _ in range(T_RE):
            m.step()
        e, p, _ = m.budget(); ends.append((e, p))
    ends = np.array(ends)
    RESTART[target] = (ends[:, 0].mean(), ends[:, 1].mean(), 1 - ends[:, 0].mean() - ends[:, 1].mean())
    s0 = SETTLE[target]
    print(f'restarted at rho = {target:g} from the {NSRC} runs that ended at rho = {1 - target:g}, after {T_RE} steps: '
          f'task-essential {RESTART[target][0]:.3f}, unrecruited {RESTART[target][2]:.3f}, against {s0[0]:.3f} and '
          f'{s0[2]:.3f} from the lattice')

start over the 30 essential sets, medians: task-essential 0.050, recruited 0.255 (from 0.150 to 0.300), unrecruited 0.695
start of seed 0, drawn as the open circle of panel C: task-essential 0.050, recruited 0.270, unrecruited 0.680
check, every condition starts from the same budget seed by seed: True
condition     task-essential           recruited                unrecruited              unrecruited below 0.05
baseline      0.230 [0.222, 0.238]     0.617 [0.593, 0.629]     0.159 [0.142, 0.168]     0.00
explore       0.233 [0.223, 0.262]     0.570 [0.566, 0.585]     0.183 [0.159, 0.202]     0.00
imitate       0.235 [0.225, 0.249]     0.601 [0.588, 0.609]     0.160 [0.153, 0.170]     0.00
both          0.391 [0.339, 0.410]     0.588 [0.557, 0.607]     0.007 [0.003, 0.037]     0.77
the other three pooled: task-essential 0.234, recruited 0.594, unrecruited 0.163
unrecruited share, both vs exploration only: U = 120.0, p = 0.0000, Cliff delta = -0.73, median difference = -0.176 [-0.196, -0.

## Step 3. The game, measured

In [6]:
med = lambda name, key: float(np.median(stack(name, key)))
GV = dict(c_O=med('both', 'c_O'), c_E=med('both', 'c_E'), a_O=med('both', 'act_O'), a_E=med('both', 'act_E'),
          w_E=med('both', 'w_E'), w_O=med('both', 'w_O'), sd_A=med('imitate', 'sd_beta'), sd_B=med('both', 'sd_beta'))
pooled = lambda name, key: stack(name, key).sum(0)
COPY, BEHIND, AHEAD6, NTOE = {}, {}, {}, {}
for rho in RHOS:
    h, k = pooled(SERIES[rho], 'h_toE'), pooled(SERIES[rho], 'k_toE')
    COPY[rho] = k.sum() / h.sum(); BEHIND[rho] = h[MID < 0].sum() / h.sum(); AHEAD6[rho] = h[MID >= 6].sum() / h.sum()
    NTOE[rho] = h.sum()
OTHER = pooled('both', 'n_other'); P_OTHER = OTHER[1] / OTHER[0]
D_B, K_B = pooled('both', 'draw'), pooled('both', 'copy')
print(f"combined condition, steps {T_HALF + 1} to {T_POP}, medians over runs: activated {GV['a_E']:.3f} of steps "
      f"(task-essential) against {GV['a_O']:.3f} (others), capability {GV['c_E']:.2f} against {GV['c_O']:.2f}, "
      f"coupling weight {GV['w_E']:.2f} (reaching the essential set) against {GV['w_O']:.2f} (elsewhere)")
print(f"spread of the direction preference at the last step: {GV['sd_B']:.3f} in the combined condition, "
      f"{GV['sd_A']:.3f} with imitation alone")
print('precision   draws of a task-essential partner   copied   partner behind   partner ahead by 6 or more')
for rho in RHOS:
    print(f'rho = {rho:<5}  {int(NTOE[rho]):>10d}                        {COPY[rho]:.3f}    {BEHIND[rho]:.3f}            '
          f'{AHEAD6[rho]:.3f}')
print(f'draws between two elements outside the essential set, combined condition: copied {P_OTHER:.3f}')
ok = D_B >= 50
dev = np.abs(K_B[ok] / D_B[ok] - 1 / (1 + np.exp(-2.0 * MID[ok]))).max()
print(f'largest gap between the observed share of copies and the Fermi rule over the bins of the combined '
      f'condition: {dev:.3f}')

combined condition, steps 2501 to 5000, medians over runs: activated 0.759 of steps (task-essential) against 0.169 (others), capability 12.18 against 2.73, coupling weight 1.97 (reaching the essential set) against 1.17 (elsewhere)
spread of the direction preference at the last step: 0.153 in the combined condition, 0.000 with imitation alone
precision   draws of a task-essential partner   copied   partner behind   partner ahead by 6 or more
rho = 0.0        129935                        0.694    0.302            0.276
rho = 0.1        259558                        0.849    0.148            0.479
rho = 0.25       528898                        0.921    0.077            0.664
rho = 0.5        724247                        0.939    0.060            0.765
rho = 1.0        932114                        0.937    0.062            0.822
draws between two elements outside the essential set, combined condition: copied 0.566
largest gap between the observed share of copies and the Fermi rule over 

## Step 4. Figure 6

In [7]:
import os
from matplotlib.transforms import Bbox

PANEL_DIR = os.environ.get('FIG_DIR', 'figs')


def save_panels(fig, stem, regions, hide=()):
    # every region of the drawn figure saved as its own panel file, PNG at 300 dpi and PDF and SVG with editable
    # text; regions maps a panel name to its box (x0, y0, x1, y1) in inches from the lower left corner of the
    # figure, and the artists in hide, which join the panels, are left out of the panel files
    os.makedirs(PANEL_DIR, exist_ok=True)
    for a in hide:
        a.set_visible(False)
    with plt.rc_context({'pdf.fonttype': 42, 'svg.fonttype': 'none'}):
        for name, box in regions.items():
            path = os.path.join(PANEL_DIR, f'{stem}{name}')
            kw = dict(bbox_inches=Bbox.from_extents(*box), facecolor='white')
            fig.savefig(path + '.png', dpi=300, metadata={'Software': None}, **kw)
            fig.savefig(path + '.pdf', metadata={'Creator': None, 'Producer': None, 'CreationDate': None}, **kw)
            fig.savefig(path + '.svg', metadata={'Creator': None, 'Date': None}, **kw)
            svg = open(path + '.svg', encoding='utf-8').read().replace('id="matplotlib.', 'id="')
            open(path + '.svg', 'w', encoding='utf-8').write(svg)
    for a in hide:
        a.set_visible(True)
    print('panel files:', ', '.join(f'{stem}{n}' for n in regions))

In [8]:
import matplotlib as mpl
from matplotlib.collections import LineCollection
from matplotlib.colors import LinearSegmentedColormap, ListedColormap, to_hex
from matplotlib.lines import Line2D
from matplotlib.patches import Circle, Rectangle

plt.rcParams.update({'font.family': 'DejaVu Sans', 'font.size': 7, 'axes.labelsize': 6.4, 'xtick.labelsize': 6.0,
                     'ytick.labelsize': 6.0, 'axes.linewidth': 0.6, 'xtick.major.width': 0.6,
                     'ytick.major.width': 0.6, 'xtick.major.size': 2.4, 'ytick.major.size': 2.4,
                     'axes.spines.top': True, 'axes.spines.right': True, 'mathtext.fontset': 'dejavusans'})
INK, NEUTRAL, GREEN = '#1A1A1A', '#37474F', '#2E7D4E'
# the color scale of Figs. 2 and 3, blue through pale yellow to red, and the part of it Fig. 6 uses
_HOT = plt.get_cmap('hot_r')(np.linspace(0.06, 0.94, 256))
_COOL = plt.get_cmap('Blues')(np.linspace(0.92, 0.06, 256))
CMAP3 = LinearSegmentedColormap.from_list('hot_div', np.vstack([_COOL, _HOT]))
LO3, HI3 = 0.14, 0.84
C_E, C_P, C_R = to_hex(CMAP3(0.665)), to_hex(CMAP3(LO3)), to_hex(CMAP3(HI3))   # task-essential, recruited, unrecruited
# the matrices of A and B run from the blue of the recruited elements through white to the red of the unrecruited,
# so that no cell takes the amber of the task-essential rows
CMAP_M = LinearSegmentedColormap.from_list('rec_unrec', [C_P, '#FFFFFF', C_R])
L_E, L_P, L_R = '#B07A00', C_P, C_R                      # the same, as text
GOLD, GOLD_T, BLUE1, GREEN1 = '#C9A227', '#8C6D0E', '#2E5090', '#2E7D4E'
SHOWN = (0.0, 0.1, 0.25, 1.0)                            # the precisions drawn in C
RAMP = dict(zip(SHOWN, (to_hex(CMAP3(0.86)), to_hex(CMAP3(0.72)), to_hex(CMAP3(0.21)), to_hex(CMAP3(0.02)))))
S3 = np.sqrt(3) / 2
DOT = (0, (1.2, 1.4))
FIG_W, FIG_H = 6.93, 3.98
NET, LIM = 1.40, 1.13
PT = NET / (2 * LIM) * 72
DMAX = np.sqrt(1.0 + S3 ** 2)
ENDS = {rho: (stack(SERIES[rho], 'E')[:, -1], stack(SERIES[rho], 'P')[:, -1]) for rho in RHOS}


def tern(e, p):
    # unrecruited at the lower left, recruited at the lower right, task-essential at the top
    return p + 0.5 * e, S3 * e


X_LAT, Y_LAT = tern(START[0], START[1])


def node_size(d):
    return 4.0 + 150.0 * ((d - 1) / (N - 2)) ** 1.15


def radius(d, dmax, dmin=2.0):
    return np.clip(np.log(dmax / np.maximum(d, dmin)) / np.log(dmax / dmin), 0, 1)


def apart(xy, rad, movable, pinned, iters=400):
    xy = xy.copy()
    for _ in range(iters):
        moved = False
        for a in movable:
            for b in range(N):
                if a == b:
                    continue
                v = xy[a] - xy[b]; dist = np.hypot(*v); need = 1.18 * (rad[a] + rad[b])
                if dist < need:
                    u = v / dist if dist > 1e-9 else np.array([1.0, 0.0])
                    if b in movable and b != pinned:
                        xy[a] += 0.5 * (need - dist) * u; xy[b] -= 0.5 * (need - dist) * u
                    else:
                        xy[a] += (need - dist) * u
                    moved = True
        if not moved:
            break
    return xy


def representative(name):
    e = stack(name, 'E')[:, -1]
    return RUNS[name][int(np.argmin(np.abs(e - np.median(e))))]


def schematic(target, n=42):
    '''Straight flow toward the target over the whole triangle, its strength proportional to the distance, as in
    the flow of the earlier figure. A drawing of the direction, not a measured speed.'''
    gx = np.linspace(0, 1, n); gy = np.linspace(0, S3, n)
    X, Y = np.meshgrid(gx, gy)
    U, V = target[0] - X, target[1] - Y
    out = Y > S3 * 2 * np.minimum(X, 1 - X) + 1e-9
    U[out] = np.nan; V[out] = np.nan
    return X, Y, U, V


def figure6(label, show=True):
    fig = plt.figure(figsize=(FIG_W, FIG_H)); fig.set_label(label)

    def axes_at(l, b, w, h):
        return fig.add_axes([l / FIG_W, b / FIG_H, w / FIG_W, h / FIG_H])

    def text_at(x, y, s, **kw):
        return fig.text(x / FIG_W, y / FIG_H, s, **kw)

    def heading(x, y, letter, title, sub=None, gap=0.07):
        t = text_at(x, y, title, fontsize=7.9, ha='center', va='bottom')
        bb = t.get_window_extent(fig.canvas.get_renderer())
        text_at(bb.x0 / fig.dpi - gap, y, letter, fontsize=9.4, fontweight='bold', color='k', ha='right', va='bottom')
        if sub:
            text_at(x, y - 0.03, sub, fontsize=6.2, ha='center', va='top')

    # ------------------------------------------------------------------ A and B
    def draw_net(l, b, run):
        W = run['W']; A = W > 0; d = A.sum(1).astype(float); isE = run['isE']
        nb = A[isE].any(axis=0) & ~isE
        th = np.pi / 2 - 2 * np.pi * np.arange(N) / N
        r = radius(d, d.max()); size = node_size(d)
        xy = np.c_[r * np.cos(th), r * np.sin(th)]
        rad = np.sqrt(size) / 2 / PT
        hub = int(np.argmax(d))
        xy = apart(xy, rad, [i for i in range(N) if d[i] > 4 and i != hub], hub)
        ax = axes_at(l, b, NET, NET)
        ax.set_xlim(-LIM, LIM); ax.set_ylim(-LIM, LIM); ax.set_aspect('equal'); ax.axis('off')
        iu = np.array(np.where(np.triu(A))).T
        ww = W[iu[:, 0], iu[:, 1]]; x = np.sqrt((ww - ww.min()) / (ww.max() - ww.min()))
        o = np.argsort(ww)
        ax.add_collection(LineCollection(xy[iu[o]], colors=plt.get_cmap('Greys')(0.22 + 0.62 * x[o]),
                                         linewidths=0.15 + 1.1 * x[o], alpha=0.8, zorder=1))
        o = np.argsort(d, kind='stable')
        ax.scatter(xy[o, 0], xy[o, 1], s=3.2 * size[o] + 10, c='#F4E58C', alpha=0.45, linewidths=0, zorder=2)
        col = np.where(isE, C_E, np.where(nb, C_P, C_R))
        ax.scatter(xy[o, 0], xy[o, 1], s=size[o], c=col[o], edgecolors=NEUTRAL, linewidths=0.3, zorder=3)
        e = np.where(isE)[0]
        ax.scatter(xy[e, 0], xy[e, 1], s=(np.sqrt(size[e]) + 3.2) ** 2, facecolors='none', edgecolors=INK,
                   linewidths=0.7, zorder=4)

    def draw_strip(l, b, w, h, name, side):
        ax = axes_at(l, b, w, h)
        e, p, rest = END[name]['E'], END[name]['P'], END[name]['R']
        for v, c in ((p, C_P), (rest, C_R), (e, C_E)):
            ax.scatter(np.ones_like(v), v, s=150, color=c, alpha=0.30, linewidths=0, zorder=3)
        lo, hi = max(0.0, min(e.min(), p.min(), rest.min())), max(e.max(), p.max(), rest.max())
        ax.set_xlim(0.0, 2.0); ax.set_ylim(lo - 0.06, hi + 0.06)
        ax.set_xticks([]); ax.set_yticks([lo, hi]); ax.set_yticklabels([f'{lo + 0.0:.2f}', f'{hi:.2f}'])
        for s_ in ('left', 'right'):
            ax.spines[s_].set_visible(False)
        for s_ in ('top', 'bottom'):
            ax.spines[s_].set_visible(True); ax.spines[s_].set_linewidth(0.7)
        if side == 'right':
            ax.yaxis.tick_right(); ax.yaxis.set_label_position('right')
        ax.tick_params(axis='y', length=0, pad=1.5)
        ax.set_ylabel(f'coupling share, step {TMAX:,}', labelpad=1.5)

    def draw_matrix(l, b, w, h, name, side):
        ax = axes_at(l, b, w, h)
        M = stack(name, 'REST').mean(0).T.astype(float)          # rank x reading
        ext = [0, TMAX, N, 0]
        ax.imshow(M, cmap=CMAP_M, vmin=0, vmax=1, aspect='auto', interpolation='nearest', extent=ext)
        ess = np.full_like(M, np.nan); ess[:5] = 1
        ax.imshow(ess, cmap=ListedColormap([C_E]), vmin=0, vmax=1, aspect='auto', interpolation='nearest', extent=ext)
        ax.set_ylim(N, 0)
        ax.set_xlim((0, TMAX) if side == 'left' else (TMAX, 0))     # practice runs inward from the outer edge
        ax.set_xticks([0, TMAX]); ax.set_xticklabels(['0', f'{TMAX:,}'])
        ax.set_yticks([])
        if side == 'right':
            ax.yaxis.tick_right(); ax.yaxis.set_label_position('right')
        for s_ in ax.spines.values():
            s_.set_visible(True); s_.set_linewidth(0.5); s_.set_linestyle(DOT)
        ax.tick_params(axis='both', length=2.0, pad=1.2)
        ax.set_xlabel('step', labelpad=0.5)
        ax.set_ylabel('elements', labelpad=2.0)

    def draw_cbar(l, b, w, h, side):
        ax = axes_at(l, b, w, h)
        ax.imshow(np.linspace(1, 0, 128)[:, None], cmap=CMAP_M, aspect='auto', extent=[0, 1, 0, 1])
        ax.set_xticks([]); ax.set_yticks([0, 1]); ax.set_yticklabels(['0', '1'])
        if side == 'right':
            ax.yaxis.tick_right(); ax.yaxis.set_label_position('right')
        for s_ in ax.spines.values():
            s_.set_visible(True); s_.set_linewidth(0.5); s_.set_linestyle(DOT)
        ax.tick_params(axis='y', length=1.6, pad=1.0, labelsize=5.6)
        ax.set_ylabel('unrecruited', fontsize=5.8, labelpad=1.5)

    def draw_key(l, b):
        ax = axes_at(l, b, 0.95, 0.62); ax.axis('off')
        hs = [Line2D([], [], ls='', marker='o', ms=8.5, color=c, alpha=0.45, mew=0) for c in (C_E, C_P, C_R)]
        ax.legend(hs, ['task-essential', 'recruited', 'unrecruited'], loc='center', frameon=False, fontsize=6.2,
                  handletextpad=0.5, labelspacing=0.75, borderpad=0.4)

    YB, YT = 2.04, 3.50
    MH = 0.66
    MX, MW = 0.24, 0.88
    CX, SX, SW, NX = 1.17, 1.86, 0.28, 2.07
    NB = YT + 0.05 - NET
    MB = YT - MH
    draw_matrix(MX, MB, MW, MH, 'imitate', 'left')
    draw_cbar(CX, MB + 0.06, 0.04, MH - 0.12, 'right')
    draw_key(MX - 0.02, YB + 0.02)
    draw_strip(SX, YB + 0.10, SW, YT - YB - 0.10, 'imitate', 'left')
    draw_net(NX, NB, representative('imitate'))
    draw_net(FIG_W - NX - NET, NB, representative('both'))
    draw_strip(FIG_W - SX - SW, YB + 0.10, SW, YT - YB - 0.10, 'both', 'right')
    draw_cbar(FIG_W - CX - 0.04, MB + 0.06, 0.04, MH - 0.12, 'left')
    draw_key(FIG_W - MX + 0.02 - 0.95, YB + 0.02)
    draw_matrix(FIG_W - MX - MW, MB, MW, MH, 'both', 'right')
    YH = 3.72
    heading(1.80, YH, 'A', 'Imitation only', r'$p_e$ = 0')
    heading(FIG_W - 1.80, YH, 'B', 'Exploration and imitation', r'$p_e$ = 0.05')
    fig.add_artist(Line2D([0.5, 0.5], [(NB + 0.04) / FIG_H, (YH + 0.14) / FIG_H], color=INK, lw=0.6, ls=DOT,
                          transform=fig.transFigure))

    # ------------------------------------------------------------------ C
    def ternary_axes(ax, labels=True):
        C = np.array([[0, 0], [1, 0], [0.5, S3]])
        for f in (0.2, 0.4, 0.6, 0.8):
            for a_, b_, c_ in ((0, 1, 2), (1, 2, 0), (2, 0, 1)):
                p0 = C[a_] + f * (C[b_] - C[a_]); p1 = C[a_] + f * (C[c_] - C[a_])
                ax.plot([p0[0], p1[0]], [p0[1], p1[1]], lw=0.35, color='#DADADA', ls=(0, (1.0, 1.6)), zorder=0)
        ax.add_patch(plt.Polygon(C, closed=True, facecolor='none', edgecolor=NEUTRAL, lw=0.6, ls=DOT, zorder=5))
        if not labels:
            return
        tl, off = 0.025, 0.27
        nL, nR, nB = np.array([-S3, 0.5]), np.array([S3, 0.5]), np.array([0.0, -1.0])
        for v in (0, 0.5, 1):
            p = np.array([(1 - v) / 2, S3 * (1 - v)])             # unrecruited, read down the left edge
            ax.plot(*np.c_[p, p + tl * nL], color=NEUTRAL, lw=0.6, zorder=5)
            ax.text(*(p + 2.6 * tl * nL), f'{v:g}', fontsize=5.8, ha='right', va='center', color='#444444')
            p = np.array([1 - v / 2, S3 * v])                     # task-essential, read up the right edge
            ax.plot(*np.c_[p, p + tl * nR], color=NEUTRAL, lw=0.6, zorder=5)
            ax.text(*(p + 2.6 * tl * nR), f'{v:g}', fontsize=5.8, ha='left', va='center', color='#444444')
            p = np.array([v, 0.0])                                # recruited, read along the bottom edge
            ax.plot(*np.c_[p, p + tl * nB], color=NEUTRAL, lw=0.6, zorder=5)
            ax.text(*(p + 2.4 * tl * nB), f'{v:g}', fontsize=5.8, ha='center', va='top', color='#444444')
        ax.text(*(np.array([0.25, S3 / 2]) + off * nL), 'unrecruited', fontsize=7.0, color=L_R, rotation=60,
                ha='center', va='center')
        ax.text(*(np.array([0.75, S3 / 2]) + off * nR), 'task-essential', fontsize=7.0, color=L_E, rotation=-60,
                ha='center', va='center')
        ax.text(0.5, -0.15, 'recruited', fontsize=7.0, color=L_P, ha='center', va='top')

    def mini_ternary(l, b, w, rho, alpha=0.62):
        K = w / 1.10
        ax = axes_at(l, b, w, K * (S3 + 0.10))
        ax.set_xlim(-0.05, 1.05); ax.set_ylim(-0.05, S3 + 0.05); ax.set_aspect('equal'); ax.axis('off')
        ax.patch.set_alpha(0.0)
        C = np.array([[0, 0], [1, 0], [0.5, S3]])
        ax.add_patch(plt.Polygon(C, closed=True, facecolor='white', alpha=0.55, edgecolor='none', zorder=0))
        ax.add_patch(plt.Polygon(C, closed=True, facecolor='none', edgecolor=NEUTRAL, lw=0.5, ls=(0, (1.0, 1.3)),
                                 zorder=5))
        e, p = ENDS[rho]
        tx, ty = tern(e.mean(), p.mean())
        X, Y, U, V = schematic((tx, ty), n=30)
        mag = np.nan_to_num(np.hypot(U, V))
        col = RAMP[rho]
        sp = ax.streamplot(X, Y, U, V, color=col, linewidth=0.25 + 0.6 * mag / DMAX, density=0.55, arrowsize=0.45,
                           arrowstyle='-|>', minlength=0.15, zorder=3)
        sp.lines.set_alpha(alpha); sp.arrows.set_alpha(alpha)
        xe, ye = tern(e, p)
        ax.scatter(xe, ye, s=2.2, color=col, alpha=0.5, linewidths=0, zorder=6)
        ax.plot(tx, ty, 'o', ms=4.2, mfc=col, mec='white', mew=0.9, zorder=7)
        ax.plot(X_LAT, Y_LAT, 'o', ms=2.8, mfc='white', mec=GREEN, mew=0.8, zorder=7)

    def draw_game(l, b, f=1.0, fs=0.98):
        '''The game of Fig. 1D drawn from the runs: element i outside the essential set, element j task-essential,
        every size and value measured over the second half of practice of the combined condition.'''
        X0, X1, Y0, Y1 = 7.15, 13.35, 3.62, 7.02
        S = 0.5 * f
        ax = axes_at(l, b, (X1 - X0) * S, (Y1 - Y0) * S)
        ax.set_xlim(X0, X1); ax.set_ylim(Y0, Y1); ax.set_aspect('equal'); ax.axis('off')
        lab = lambda x, y, t, **k: ax.text(x, y, t, **{**dict(fontsize=6.0 * fs, ha='center', va='center', color=INK),
                                                       **k})
        def arrow(xy, xytext, color, lw, ms):
            ax.annotate('', xy=xy, xytext=xytext, zorder=5, arrowprops=dict(arrowstyle='-|>', lw=lw * f, color=color,
                                                                           mutation_scale=ms * f, shrinkA=0, shrinkB=0))
        xi, xj, yn = 8.05, 12.45, 5.3
        xm = 0.5 * (xi + xj)
        fx0, fx1, fy0, fy1 = xi + 0.55, xj - 0.55, yn - 0.95, yn + 0.95
        XR = 6.0
        fx = lambda dc: fx0 + (np.clip(dc, -XR, XR) + XR) / (2 * XR) * (fx1 - fx0)
        fy = lambda p: fy0 + p * (fy1 - fy0)
        for p in (0, 1):                                  # top and bottom only, no side lines
            ax.plot([fx0, fx1], [fy(p)] * 2, lw=0.6 * f, color=INK, ls=DOT, zorder=2)
        ax.plot([fx(0)] * 2, [fy0, fy1], lw=0.6 * f, color=INK, ls=DOT, zorder=2)
        dc = np.linspace(-XR, XR, 300)
        ax.plot(fx(dc), fy(1 / (1 + np.exp(-2.0 * dc))), '-', lw=1.0 * f, color=NEUTRAL, zorder=3)
        ok = (np.abs(MID) <= XR) & (D_B >= 50)
        ax.scatter(fx(MID[ok]), fy(K_B[ok] / D_B[ok]), s=4.0 * f, color=INK, zorder=4, linewidths=0)
        lab(fx0 - 0.06, fy(1), '1', fontsize=5.0 * fs, ha='right', color='#555555')
        lab(fx0 - 0.06, fy(0), '0', fontsize=5.0 * fs, ha='right', color='#555555')
        lab(xm, fy0 - 0.16, r'$c_j - c_i$', fontsize=5.8 * fs, color='#333333')
        lab(fx(-3.6), fy(0.12), 'imitation', fontsize=5.4 * fs, color=NEUTRAL, style='italic')
        # the start of practice: every element begins with the same capability, so c_j - c_i = 0 and the copy is
        # an even chance, drawn as the open circle that marks the lattice in the ternary panels
        ax.scatter([fx(0.0)], [fy(0.5)], s=22 * f, facecolors='white', edgecolors=GREEN, linewidths=1.1 * f, zorder=6)
        lab(fx(0.0) - 0.13, fy(0.5), 'start', fontsize=5.0 * fs, ha='right', va='center', color=GREEN)
        # the observed share of draws of a task-essential partner that ended in a copy, placed on the curve
        for rho in SHOWN:
            p = COPY[rho]; x = np.log(p / (1 - p)) / 2.0
            ax.scatter([fx(x)], [fy(p)], s=26 * f, color=RAMP[rho], edgecolors='white', linewidths=0.7, zorder=6)
        for rho, txt, side in ((0.0, r'$\rho$ = 0', -1), (0.1, '0.1', 1), (1.0, '0.25, 1', 1)):
            p = COPY[rho]; x = np.log(p / (1 - p)) / 2.0
            lab(fx(x) + side * 0.13, fy(p) - 0.01, txt, fontsize=5.0 * fs, ha='left' if side > 0 else 'right',
                va='top', color=RAMP[rho])
        # the two middle precisions of C, small and see-through, in the empty corners of the curve
        tw = 1.22
        mini_ternary(l + (fx0 + 0.06 - X0) * S, b + (fy(0.36) - Y0) * S, tw * S, 0.1)
        mini_ternary(l + (fx1 - 0.06 - tw - X0) * S, b + (fy(0.10) - Y0) * S, tw * S, 0.25)
        lab(fx0 + 0.06 + tw / 2, fy(0.36) - 0.05, r'($\rho$ $\rightarrow$ 0.1)', fontsize=5.0 * fs, va='top',
            color=RAMP[0.1])
        lab(fx1 - 0.06 - tw / 2, fy(0.10) + tw * (S3 + 0.10) / 1.10 + 0.02, r'($\rho$ $\rightarrow$ 0.25)',
            fontsize=5.0 * fs, va='bottom', color=RAMP[0.25])
        # the two elements, capability as the gold core and the bar, both to scale
        for (x, y, c, name, flip, ess) in ((xi, yn, GV['c_O'], 'i', False, False), (xj, yn, GV['c_E'], 'j', True, True)):
            for ang in ((150, 180, 210) if not flip else (-30, 0, 30)):
                t = np.radians(ang)
                x1, y1, x2, y2 = x + 0.31 * np.cos(t), y + 0.31 * np.sin(t), x + 0.66 * np.cos(t), y + 0.66 * np.sin(t)
                ax.plot([x1, x2], [y1, y2], color=NEUTRAL, lw=0.8 * f, solid_capstyle='round', zorder=3)
                for sub in (-22, 22):
                    t2 = np.radians(ang + sub)
                    ax.plot([x2, x2 + 0.18 * np.cos(t2)], [y2, y2 + 0.18 * np.sin(t2)], color=NEUTRAL, lw=0.6 * f,
                            zorder=3)
            ax.add_patch(Circle((x, y), 0.31, facecolor='white', edgecolor=INK if ess else NEUTRAL,
                                lw=(1.3 if ess else 0.8) * f, zorder=4))
            ax.add_patch(Circle((x, y), 0.05 * np.sqrt(c), facecolor=GOLD, edgecolor='none', zorder=5))
            ax.text(x + (0.2 if flip else -0.2), y + 0.46, name, fontsize=7.6 * fs, style='italic',
                    ha='left' if flip else 'right', va='bottom', zorder=6)
            h = 0.068 * c
            ax.add_patch(Rectangle((x - 0.12, y + 0.44), 0.24, h, facecolor=GOLD, alpha=0.9, edgecolor='none'))
            lab(x, y + 0.44 + h + 0.05, f'${{c_{name}}}$ {c:.1f}', fontsize=6.0 * fs, va='bottom')
        lab(xi, yn - 0.42, 'non-essential', fontsize=5.0 * fs, color='#555555', va='top')
        lab(xj, yn - 0.42, 'task-essential', fontsize=5.0 * fs, color=GOLD_T, va='top')
        ax.plot([xi + 0.31, fx0], [yn, yn], color=BLUE1, lw=1.4 * f, solid_capstyle='round', zorder=1)
        ax.plot([fx1, xj - 0.31], [yn, yn], color=BLUE1, lw=1.4 * f, solid_capstyle='round', zorder=1)
        lab(xm, fy1 + 0.10, r'coupling $w_{ij}$ ' + f"{GV['w_E']:.1f}, elsewhere {GV['w_O']:.1f}", fontsize=5.8 * fs,
            va='bottom', color=BLUE1)
        for x_, a_ in ((xi, GV['a_O']), (xj, GV['a_E'])):
            arrow((x_, yn - 0.62), (x_, yn - 1.20), GOLD, 0.5 + 2.6 * a_, 7)
        lab(xi, yn - 1.30, f"activated {GV['a_O']:.2f}", fontsize=5.6 * fs, color=GOLD_T, va='top')
        lab(xj, yn - 1.30, rf"activated {GV['a_E']:.2f}, $\rho$", fontsize=5.6 * fs, color=GOLD_T, va='top')
        lab(xi, yn - 1.52, r'explore $\beta_i$, sd ' + f"{GV['sd_B']:.2f} in B, {GV['sd_A']:.0f} in A",
            fontsize=5.0 * fs, color=GREEN1, va='top')
        lab(xm, fy0 - 0.40, f'measured in B, steps {T_HALF:,} to {TMAX:,}', fontsize=5.0 * fs, color='#777777',
            style='italic')

    BASE, KB = 0.38, 1.21
    XB, YBL = (-0.26, 1.26), (-0.30, S3 + 0.06)
    WB = KB * (XB[1] - XB[0])
    GW = (13.35 - 7.15) * 0.5
    draw_game(FIG_W / 2 - GW / 2, 0.10)
    heading(FIG_W / 2, 1.76, 'C', 'Functional capability steers where the coupling settles')
    for k, rho in ((0, 0.0), (1, 1.0)):
        col = RAMP[rho]
        left = 0.0 if k == 0 else FIG_W - WB
        ax = axes_at(left, BASE + KB * YBL[0], WB, KB * (YBL[1] - YBL[0]))
        ax.set_aspect('equal'); ax.axis('off'); ax.set_xlim(*XB); ax.set_ylim(*YBL)
        ternary_axes(ax, labels=True)
        e, p = ENDS[rho]
        tx, ty = tern(e.mean(), p.mean())
        X, Y, U, V = schematic((tx, ty))
        mag = np.nan_to_num(np.hypot(U, V))
        ax.streamplot(X, Y, U, V, color=col, linewidth=0.35 + 1.35 * mag / DMAX, density=1.05, arrowsize=0.85,
                      arrowstyle='-|>', minlength=0.12, zorder=3)
        xe, ye = tern(e, p)
        ax.scatter(xe, ye, s=5.0, color=col, alpha=0.55, linewidths=0, zorder=6)
        ax.plot(tx, ty, 'o', ms=7.2, mfc=col, mec='white', mew=1.3, zorder=7)
        ax.plot(X_LAT, Y_LAT, 'o', ms=4.4, mfc='white', mec=GREEN, mew=1.1, zorder=7)
        apex = BASE + KB * S3
        text_at(left + KB * (0.5 - XB[0]), apex + 0.17, r'($\rho$ $\rightarrow$ {:g})'.format(rho), fontsize=7.0,
                color=col, ha='center', va='bottom')
        if k == 0:
            text_at(left + 0.20, apex + 0.56, 'coupling stays\nunrecruited', fontsize=7.2, color=col, ha='left',
                    va='top', linespacing=0.95)
        else:
            text_at(left + WB - 0.20, apex + 0.56, 'coupling recruited\nto the task', fontsize=7.2, color=col,
                    ha='right', va='top', linespacing=0.95)
    if not show:
        return fig
    plt.show()


figure6('Manuscript Fig 6')
figure6('Manuscript Fig 6A-C')

# each panel as its own file, cut from one more drawing of the same figure, as Figs. 2 to 4 are saved panel by
# panel: A and B above, and below the left end of the precision series, the measured game with the heading of C
# and the right end
fig = figure6('Fig 6 for the panel files', show=False)
WB = 1.21 * (1.26 + 0.26)                          # the width of each triangle of C, as in figure6
letter_C = [t for t in fig.texts if t.get_text() == 'C'][0]
XL = min(WB, letter_C.get_window_extent(fig.canvas.get_renderer()).x0 / fig.dpi - 0.04)
XR, YS = FIG_W - WB, 2.02                          # YS lies between the keys of A and B and the labels of C
save_panels(fig, 'Manuscript_Fig_6', {'A': (0.0, YS, FIG_W / 2 - 0.01, FIG_H),
                                      'B': (FIG_W / 2 + 0.01, YS, FIG_W, FIG_H),
                                      'C_left': (0.0, 0.0, XL, YS),
                                      'C_middle': (XL, 0.0, XR, YS),
                                      'C_right': (XR, 0.0, FIG_W, YS)})
plt.close(fig)
print('Figure 6 drawn: imitation alone (A) and with exploration (B) as matrix, budget and network, and where the '
      'coupling settles as precision rises around the measured game (C)')

panel files: Manuscript_Fig_6A, Manuscript_Fig_6B, Manuscript_Fig_6C_left, Manuscript_Fig_6C_middle, Manuscript_Fig_6C_right
Figure 6 drawn: imitation alone (A) and with exploration (B) as matrix, budget and network, and where the coupling settles as precision rises around the measured game (C)


## Step 5. Values for the manuscript, and checks

In [9]:
fmt = lambda v, f: format(v, f).replace('-', '−')
pfmt = lambda p: 'P < 0.001' if p < 0.001 else f'P = {p:.3f}'
OUT = {'F6_T': f'{TMAX:,}', 'F6_E0': f'{START_MED[0]:.2f}', 'F6_P0': f'{START_MED[1]:.2f}',
       'F6_R0': f'{START_MED[2]:.2f}', 'F6_P0_LO': f'{P0.min():.2f}', 'F6_P0_HI': f'{P0.max():.2f}',
       'F6_THALF': f'{T_HALF:,}'}
for name, key in (('baseline', 'BASE'), ('explore', 'EXP'), ('imitate', 'IMI'), ('both', 'BOTH')):
    for part in ('E', 'P', 'R'):
        v = END[name][part]; lo, hi = boot_median(v)
        OUT[f'F6_{part}_{key}'] = f'{np.median(v):.3f}'
        OUT[f'F6_{part}_{key}_CI'] = f'[{lo:.3f},_{hi:.3f}]'
    OUT[f'F6_R05_FRAC_{key}'] = f'{(END[name]["R"] < 0.05).mean():.2f}'
OUT['F6_R_REST'] = f'{np.median(rest3("R")):.3f}'
OUT['F6_E_REST'] = f'{np.median(rest3("E")):.3f}'
OUT['F6_T_R05'] = f'{T_R05:,}' if T_R05 else 'none'
OUT['F6_T_E02'] = f'{T_E02:,}' if T_E02 else 'none'
for t, key in ((T_REST_EXP, 'REXP'), (T_REST_BASE, 'RBASE'), (T_PART_EXP, 'PEXP')):
    OUT[f'F6_T{key}_P'] = pfmt(t['p']).replace(' ', '_')
    OUT[f'F6_T{key}_D'] = fmt(t['d'], '+.2f')
    OUT[f'F6_T{key}_U'] = f"{t['U']:.1f}"
    OUT[f'F6_T{key}_DIFF'] = fmt(t['diff'], '+.3f')
    OUT[f'F6_T{key}_CI'] = '[' + fmt(t['lo'], '+.3f') + ',_' + fmt(t['hi'], '+.3f') + ']'
for rho in RHOS:
    tag = f'{rho:g}'.replace('.', '')
    OUT[f'F6_S{tag}_E'], OUT[f'F6_S{tag}_P'], OUT[f'F6_S{tag}_R'] = (f'{v:.2f}' for v in SETTLE[rho])
    OUT[f'F6_COPY{tag}'] = f'{COPY[rho]:.2f}'
for target, tag in ((0.0, 'TO0'), (1.0, 'TO1')):
    OUT[f'F6_RS_{tag}_E'], _, OUT[f'F6_RS_{tag}_R'] = (f'{v:.2f}' for v in RESTART[target])
OUT['F6_RS_T'] = f'{T_RE:,}'
OUT['F6_RS_N'] = f'{NSRC}'
OUT['F6_G_CO'], OUT['F6_G_CE'] = f"{GV['c_O']:.1f}", f"{GV['c_E']:.1f}"
OUT['F6_G_AO'], OUT['F6_G_AE'] = f"{GV['a_O']:.2f}", f"{GV['a_E']:.2f}"
OUT['F6_G_WE'], OUT['F6_G_WO'] = f"{GV['w_E']:.1f}", f"{GV['w_O']:.1f}"
OUT['F6_G_SDB'], OUT['F6_G_SDA'] = f"{GV['sd_B']:.2f}", f"{GV['sd_A']:.0f}"
OUT['F6_G_BEHIND0'], OUT['F6_G_BEHIND1'] = f'{BEHIND[0.0]:.2f}', f'{BEHIND[1.0]:.2f}'
OUT['F6_G_AHEAD0'], OUT['F6_G_AHEAD1'] = f'{AHEAD6[0.0]:.2f}', f'{AHEAD6[1.0]:.2f}'
OUT['F6_G_NRATIO'] = f'{NTOE[1.0] / NTOE[0.0]:.1f}'
OUT['F6_G_POTHER'] = f'{P_OTHER:.2f}'
OUT['F6_G_FERMIGAP'] = f'{dev:.3f}'
for k, v in OUT.items():
    print(f'{k} = {v}')
print('check, a read run equals the same seed run straight through:', CHECK_PLAIN)
print('check, the budget read from the couplings equals the measures of the model at the last step:', CHECK_SAME)
print('check, the shares sum to one in every reading:',
      bool(all(np.allclose(r['E'] + r['P'] + r['R'], 1.0) for n in RUNS for r in RUNS[n])))

F6_T = 5,000
F6_E0 = 0.05
F6_P0 = 0.26
F6_R0 = 0.69
F6_P0_LO = 0.15
F6_P0_HI = 0.30
F6_THALF = 2,500
F6_E_BASE = 0.230
F6_E_BASE_CI = [0.222,_0.238]
F6_P_BASE = 0.617
F6_P_BASE_CI = [0.593,_0.629]
F6_R_BASE = 0.159
F6_R_BASE_CI = [0.142,_0.168]
F6_R05_FRAC_BASE = 0.00
F6_E_EXP = 0.233
F6_E_EXP_CI = [0.223,_0.262]
F6_P_EXP = 0.570
F6_P_EXP_CI = [0.566,_0.585]
F6_R_EXP = 0.183
F6_R_EXP_CI = [0.159,_0.202]
F6_R05_FRAC_EXP = 0.00
F6_E_IMI = 0.235
F6_E_IMI_CI = [0.225,_0.249]
F6_P_IMI = 0.601
F6_P_IMI_CI = [0.588,_0.609]
F6_R_IMI = 0.160
F6_R_IMI_CI = [0.153,_0.170]
F6_R05_FRAC_IMI = 0.00
F6_E_BOTH = 0.391
F6_E_BOTH_CI = [0.339,_0.410]
F6_P_BOTH = 0.588
F6_P_BOTH_CI = [0.557,_0.607]
F6_R_BOTH = 0.007
F6_R_BOTH_CI = [0.003,_0.037]
F6_R05_FRAC_BOTH = 0.77
F6_R_REST = 0.163
F6_E_REST = 0.234
F6_T_R05 = 3,825
F6_T_E02 = 2,200
F6_TREXP_P = P_<_0.001
F6_TREXP_D = −0.73
F6_TREXP_U = 120.0
F6_TREXP_DIFF = −0.176
F6_TREXP_CI = [−0.196,_−0.135]
F6_TRBASE_P = P_<_0.001
F6_TRBASE_D = −0.68
F6_TRBASE_U 